# Week 3-2 · bounded review loop와 종료 보장

## 시나리오
첫 proposal이 위험하면 commander로 되돌려 수정하되, `max_revisions`를 소진하면 `fail_closed`로 종료하는 graph를 만듭니다.

## 학습 목표
- 조건부 edge로 review 결과에 따라 loop 또는 종료한다.
- `revision_count`와 `max_revisions`로 반복 상한을 둔다.
- 안전한 수정과 계속 위험한 수정을 모두 실행한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · loop state와 node

In [ ]:
# 실행 순서: 1단계 · loop state와 node에서 ReviewState, commander_node, review_node을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · loop state와 node.
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

# proposal 수정 횟수와 상한을 state에 넣어 loop 종료를 보장합니다.
class ReviewState(TypedDict):
    always_unsafe: bool
    proposal: str
    revision_count: int
    max_revisions: int
    status: str
    steps: list[str]

# commander 역할이 실행하지 않을 read-only proposal만 작성합니다.
def commander_node(state: ReviewState) -> dict:
    revised = state["revision_count"] > 0
    if state["always_unsafe"] or not revised:
        proposal = "kubectl delete pod checkout"
    else:
        proposal = "kubectl get pods"
    return {"proposal": proposal, "steps": state["steps"] + ["commander"]}

# proposal을 결정적 prefix로 검사해 approved 또는 unsafe로 표시합니다.
def review_node(state: ReviewState) -> dict:
    safe = state["proposal"].startswith("kubectl get ")
    status = "approved" if safe else "unsafe"
    return {"status": status, "steps": state["steps"] + ["review"]}

# 재검토 횟수를 한 번 증가시켜 무한 loop를 막는 근거를 남깁니다.
def revise_node(state: ReviewState) -> dict:
    return {"revision_count": state["revision_count"] + 1, "steps": state["steps"] + ["revise"]}

# 수정 상한을 소진하면 성공으로 가장하지 않고 fail_closed로 종료합니다.
def fail_closed(state: ReviewState) -> dict:
    return {"status": "fail_closed", "steps": state["steps"] + ["fail_closed"]}

# 승인 여부와 남은 revision budget으로 종료·수정·차단 edge를 고릅니다.
def choose_review_path(state: ReviewState) -> str:
    if state["status"] == "approved":
        return "finish"
    if state["revision_count"] < state["max_revisions"]:
        return "revise"
    return "fail_closed"

### 2단계 · conditional loop 연결

In [ ]:
# 실행 순서: 2단계 · conditional loop 연결에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · conditional loop 연결.
practice_builder = StateGraph(ReviewState)
for name, node in {"commander": commander_node, "review": review_node, "revise": revise_node, "fail_closed": fail_closed}.items():
    practice_builder.add_node(name, node)
practice_builder.add_edge(START, "commander")
practice_builder.add_edge("commander", "review")
practice_builder.add_conditional_edges("review", choose_review_path, {
    "finish": END,
    "revise": "revise",
    "fail_closed": "fail_closed",
})
practice_builder.add_edge("revise", "commander")
practice_builder.add_edge("fail_closed", END)
practice_graph = practice_builder.compile()

### 3단계 · 성공 수정과 한도 소진 비교

In [ ]:
# 실행 순서: 3단계 · 성공 수정과 한도 소진 비교에서 practice_review을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 성공 수정과 한도 소진 비교.
# 같은 graph에 수정 가능/불가능 fixture를 넣어 두 종료 경로를 비교합니다.
def practice_review(always_unsafe: bool) -> ReviewState:
    return practice_graph.invoke({
        "always_unsafe": always_unsafe, "proposal": "", "revision_count": 0,
        "max_revisions": 1, "status": "", "steps": [],
    })

revised_safe = practice_review(False)
blocked = practice_review(True)
assert revised_safe["status"] == "approved" and revised_safe["revision_count"] == 1
assert blocked["status"] == "fail_closed" and blocked["revision_count"] == blocked["max_revisions"]
assert blocked["steps"].count("commander") == 2
{"revised_safe": revised_safe, "blocked": blocked}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
상한을 넘겨 계속 재시도하지 않습니다. 안전성을 증명하지 못하면 성공처럼 끝내지 않고 `fail_closed`로 종료합니다.

## 실제 app 연결
Week 3 app의 `commander → risk_guard → revise/finish/blocked` 구조를 축소했습니다. Week 2의 1회 routing과 달리 이번 conditional edge는 이전 node로 돌아가는 bounded loop입니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `03_command_safety_human_approval.ipynb`에서는 loop가 만든 proposal을 결정적 정책과 사람 승인 경계로 검사합니다.